# Building RAG system to store and retrieve Pokemon Data

## Introduction
As silly Pokemon nerds, we want to collect info about Pokemon easily and from a good source. The problem is the API is not that handy for querying, and sites elsewhere are not always accurate or reliable.</br></br>
Our goal is to create a queryable RAG (Retrieval-Augmented Generation) system of reliable Pokemon data in these steps:
<ol>
<li>Define a series of documents by key feature type -- species, moves, generation, etc.</li>
<li>Load documents into a FAISS Vector Store to enable Similarity Search</li>
<li>Create the search tool to respond to query</li>
<li>Build the JSON query planner to turn a user query into an LLM input.</li>
</ol>

### Setup
**Includes, key parameters and API connection objects**

In [2]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pathlib import Path
import os, json, re, requests
from tqdm import tqdm
from pydantic import BaseModel, Field
from langchain_core.tools import tool
import numpy as np, requests
import faiss, json
from typing import List, Dict, Any, Optional, Callable, Literal
import nltk
from functools import lru_cache
import concurrent.futures
from langchain_ollama import ChatOllama
import re, unicodedata
from pokemon_mtg_ref.pokemon_mtg import GEN_TO_REGION, REGION_TO_GEN, TYPE_SET, SHAPES, ICONIC_MOVES

# Set to true to rebuild database of Pokemon info
BUILD_INDEX = False
CACHE = Path("./datasets/poke_cache"); CACHE.mkdir(exist_ok=True)
INDEX_PATH = "datasets/poke_faiss"

# LLM info
OLLAMA_URL = os.environ.get("OLLAMA_EMBED_URL", "http://localhost:11434/api/embeddings")
EMBED_MODEL = os.environ.get("OLLAMA_EMBED_MODEL", "nomic-embed-text")
LLM = ChatOllama(model="gpt-oss:20b", temperature=0.2)
PLANNER = ChatOllama(model="gpt-oss:20b", temperature=0.0)

# API parameters
BASE  = "https://pokeapi.co/api/v2"

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "poke-rag/1.0",
    "Accept-Encoding": "gzip, deflate",
})

retry = Retry(
    total=5,
    connect=3,
    read=3,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)
adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
SESSION.mount("https://", adapter)
SESSION.mount("http://", adapter)

def _get(url: str, *, timeout=(5, 20)):
    r = SESSION.get(url, timeout=timeout)
    if r.status_code == 404:
        # raise a light error the caller can catch quickly
        raise FileNotFoundError(f"404 for {url}")
    r.raise_for_status()
    return r.json()

def _get_or_none(url: str, *, timeout=(5,20)):
    try:
        return _get(url, timeout=timeout)
    except FileNotFoundError:
        return None


**Pokemon constants**
These are Pokemon related values that change very rarely. Define as constants for reference.

**JSON Parsing** Parse JSON strings from the Pokemon API into features for the RAG.

In [3]:

# General-purpose helper functions

def _roman(n:int)->str:
    r = ["","I","II","III","IV","V","VI","VII","VIII","IX","X",
         "XI","XII","XIII","XIV","XV","XVI","XVII","XVIII","XIX","XX"]
    return r[n] if 0 <= n < len(r) else str(n)

def _meters(decimeters:int) -> float: return round(decimeters / 10.0, 2)
def _kg(hectograms:int) -> float:     return round(hectograms / 10.0, 1)

def _size_class(height_m: float) -> str:
    if height_m < 0.8: return "small"
    if height_m <= 1.5: return "medium"
    return "large"

def _cap(s: str) -> str:
    return (s or "").replace("-", " ").title()

def _session():
    s = requests.Session()
    s.headers.update({"User-Agent":"poke-rag/1.0"})
    return s

def _cache_get(name):
    f = CACHE/f"{name}.json"
    return json.load(open(f)) if f.exists() else None

def _cache_put(name, obj):
    f = CACHE/f"{name}.json"
    with open(f,"w") as fp: json.dump(obj, fp)

def _is_legend(q: str): return bool(re.search(r"\blegendary\b", q.lower()))

def _is_mythic(q: str): return bool(re.search(r"\bmythic(?:al)?\b", q.lower()))

def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()

def detect_region(text: str):
    ql = text.lower()
    for r in REGION_TO_GEN:
        if re.search(rf"\b{r}\b", ql): return r
    return None

def detect_types(text: str):
    ql = text.lower()
    return [t.title() for t in TYPE_SET if re.search(rf"\b{t}\b", ql)]

def get_json(url, name=None, timeout=15):
    if name:
        c = _cache_get(name)
        if c is not None: return c
    s = _session()
    r = s.get(url, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if name: _cache_put(name, data)
    return data

def resolve_species_and_default_form_sync(slug: str):
    nid = slug.strip().lower()

    # 1) Try as Pokémon form/name first (fast path for forms)
    pk = _get_or_none(f"{BASE}/pokemon/{nid}")
    if pk is not None:
        species_name = (pk.get("species") or {}).get("name")
        sp = _get(f"{BASE}/pokemon-species/{species_name}")
        # ensure we return the **default** form
        varieties = sp.get("varieties") or []
        def_form = next((v["pokemon"]["name"] for v in varieties if v.get("is_default")), pk.get("name"))
        if pk.get("name") != def_form:
            pk = _get(f"{BASE}/pokemon/{def_form}")
        return sp, pk, species_name, def_form

    # 2) Otherwise, treat as species
    sp = _get_or_none(f"{BASE}/pokemon-species/{nid}")
    if sp is None:
        raise RuntimeError(f"Unknown Pokémon or species: '{slug}'")

    varieties = sp.get("varieties") or []
    def_form = next((v["pokemon"]["name"] for v in varieties if v.get("is_default")),
                    (varieties[0].get("pokemon") or {}).get("name") if varieties else sp["name"])
    pk = _get(f"{BASE}/pokemon/{def_form}")
    return sp, pk, sp["name"], def_form

def metrics_from_pokemon(pk_json):
    """Height in meters, weight in kilograms from a Pokémon JSON (decimeters / hectograms)."""
    h_dm = pk_json.get("height")
    w_hg = pk_json.get("weight")
    return ((h_dm / 10.0) if h_dm is not None else None,
            (w_hg / 10.0) if w_hg is not None else None)

def list_all(resource):
    # e.g. resource="pokemon-species"
    key = f"index_{resource}"
    cached = _cache_get(key)
    if cached: return cached["results"]
    url = f"{BASE}/{resource}?limit=2000"
    data = get_json(url)
    _cache_put(key, data)
    return data["results"]

def normalize_attr(a: str | None) -> str | None:
    if not a: return a
    a = a.lower().strip().replace(" ", "_")
    aliases = {
        "effect": "effect_text",
        "damageclass": "damage_class",
        # plural ↔ singular harmonization
        "shapes": "shape",
        "regions": "region",
        "generations": "generation",
        "type": "types",
        # colloquial
        "height": "height_m",
        "weight": "weight_kg",
    }
    return aliases.get(a, a)


def pull_many(index, namer, workers=16):
    s = _session()
    def pull(rec):
        r = s.get(rec["url"], timeout=15); r.raise_for_status()
        data = r.json()
        _cache_put(namer(data), data)
        return data
    out=[]
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
        for d in tqdm(ex.map(pull, index), total=len(index), desc="pull"):
            out.append(d)
    return out


### RAG Pipeline Generation
#### Define documents
Create a series of documents by feature type to streamline search.
<ul><li>Species</li>
<li>Move</li>
<li>Abilities</li>
<li>Type</li>
<li>Region</li>
<li>Generation</li></ul>

In [4]:
# Query and load data into documents

def move_effect_en(move) -> str:
    # pick English effect; prefer short_effect
    en = next((e for e in move.get("effect_entries", []) if e["language"]["name"]=="en"), None)
    txt = (en.get("short_effect") or en.get("effect") or "").replace("\f"," ").strip() if en else ""
    # substitute $effect_chance with the numeric value if present
    chance = move.get("effect_chance")
    if chance is not None:
        txt = txt.replace("$effect_chance", str(chance))
    # normalize whitespace
    return re.sub(r"\s+"," ", txt)

# Clean the flavor text for a pokemon
def clean_en(entries, key="language"):
    for e in entries:
        lang = e.get(key, {}).get("name")
        if lang == "en":
            # PokeAPI flavor/effect texts have \n/\f and strange spacing
            return re.sub(r"\s+", " ", (e.get("flavor_text") or e.get("short_effect") or e.get("effect") or "").replace("\f"," ")).strip()
    return ""

# Identify the default type for a pokemon species
def default_types_for_species(species_json):
    for v in species_json.get("varieties", []):
        if v.get("is_default") and v.get("pokemon"):
            p = get_json(v["pokemon"]["url"], name=f"pokemon_{v['pokemon']['name']}")
            return [t["type"]["name"] for t in sorted(p["types"], key=lambda x: x["slot"])]
    return []

def move_doc(m):
    mid = m["id"]; name=m["name"]
    type_ = (m.get("type") or {}).get("name")
    dmg = (m.get("damage_class") or {}).get("name")
    power, acc, pp = m.get("power"), m.get("accuracy"), m.get("pp")
    eff_text = move_effect_en(m)
    main = {
        "id": mid, "kind":"move","name":name,"type":type_,"damage_class":dmg,
        "power":power,"accuracy":acc,"pp":pp, "effect_text": eff_text   
    }
    text = f"Move {name} — Type: {type_}. Class: {dmg}. Power: {power}. Accuracy: {acc}. PP: {pp}. Effect: {eff_text}"
    return {"id": f"move:{mid}", "name": name, "text": text, "meta": main}


def _default_pokemon_name_from_species(s: dict) -> str | None:
    # PokéAPI species JSON has 'varieties' with 'is_default' flag
    for v in s.get("varieties", []) or []:
        if v.get("is_default") and v.get("pokemon", {}).get("name"):
            return v["pokemon"]["name"]
    # Fallback: first variety
    if s.get("varieties"):
        return (s["varieties"][0].get("pokemon") or {}).get("name")
    return None

def species_doc(
    s: dict,
    *,
    pokemon_lookup: Callable | None = None  # pokemon_lookup(name:str) -> dict (raw or ETL'd)
) -> dict:
    name = s["name"]; sid = s["id"]
    shape = (s.get("shape") or {}).get("name")
    color = (s.get("color") or {}).get("name")
    genus = next((g["genus"] for g in s.get("genera", []) if g["language"]["name"] == "en"), None)
    flavor = clean_en(s.get("flavor_text_entries", []))
    gen = (s.get("generation") or {}).get("name")
    types = default_types_for_species(s)  # you already had this

    # --- NEW: pull metrics from the default Pokémon form if possible ---
    height_m = weight_kg = None
    if pokemon_lookup:
        poke_name = _default_pokemon_name_from_species(s)
        if poke_name:
            poke = pokemon_lookup(poke_name)  # implement outside: fetch from cache or API
            height_m, weight_kg = metrics_from_pokemon(poke) 

    meta = {
        "id": sid, "kind": "species", "name": name,
        "shape": shape, "color": color, "genus": genus,
        "is_mythical": s["is_mythical"], "is_legendary": s["is_legendary"],
        "generation": gen, "types": types,
        # --- NEW fields (present only if we found them) ---
        "height_m": height_m, "weight_kg": weight_kg,
        # Optional: store region too for easier facetting later
        "main_region": GEN_TO_REGION.get(gen) if gen else None,
    }

    # Build human text (include metrics if available)
    parts = [
        f"{name.capitalize()} — {genus or 'Pokemon'}.",
        f"Types: {', '.join(types) or 'unknown'}.",
        f"Shape: {shape}.", f"Color: {color}.",
        f"Legendary: {s['is_legendary']}.", f"Mythical: {s['is_mythical']}.",
        f"Generation: {gen}.",
    ]
    if height_m is not None: parts.append(f"Height: {height_m:.2f} m.")
    if weight_kg is not None: parts.append(f"Weight: {weight_kg:.1f} kg.")
    parts.append(f"Flavor: {flavor}")

    text = " ".join(parts)
    return {"id": f"species:{sid}", "name": name, "text": text, "meta": meta}

def pokemon_doc(p, species_lookup):
    name = p["name"]; pid = p["id"]
    types = [t["type"]["name"] for t in sorted(p["types"], key=lambda x:x["slot"])]
    abilities = [a["ability"]["name"] for a in p["abilities"]]
    moves = [m["move"]["name"] for m in p["moves"][:12]]
    h_m, w_kg = p["height"]/10.0, p["weight"]/10.0
    sp = species_lookup.get(name)
    main = {
        "id": pid, "kind":"pokemon", "name": name, "types": types,
        "height_m": h_m, "weight_kg": w_kg, "abilities": abilities, "moves": moves,
        "is_mythical": sp.get("is_mythical") if sp else None, "is_legendary": sp.get("is_legendary") if sp else None,
        "generation": sp.get("generation") if sp else None
    }
    text = f"{name.capitalize()} — Types: {', '.join(types)}. Abilities: {', '.join(abilities)}. " \
           f"Size: {h_m} m, {w_kg} kg. Common moves: {', '.join(moves)}. " \
           f"Legendary: {main['is_legendary']}. Mythical: {main['is_mythical']}."
    return {"id": f"pokemon:{pid}", "name": name, "text": text, "meta": main}


def ability_doc(a):
    aid=a["id"]; name=a["name"]
    effect = clean_en(a.get("effect_entries", []))
    short  = clean_en(a.get("effect_entries", []))
    main = {"id":aid,"kind":"ability","name":name}
    text = f"Ability {name} — {short or effect}"
    return {"id": f"ability:{aid}", "name": name, "text": text, "meta": main}

def type_doc(t):
    tid = t["id"]; name=t["name"]
    rel = t.get("damage_relations", {})
    def names(xs): return ", ".join(e["name"] for e in xs) or "none"
    text = (f"Type {name} — double_damage_to: {names(rel.get('double_damage_to',[]))}; "
            f"double_damage_from: {names(rel.get('double_damage_from',[]))}; "
            f"half_damage_to: {names(rel.get('half_damage_to',[]))}; "
            f"half_damage_from: {names(rel.get('half_damage_from',[]))}; "
            f"no_damage_to: {names(rel.get('no_damage_to',[]))}; "
            f"no_damage_from: {names(rel.get('no_damage_from',[]))}.")
    return {"id": f"type:{tid}", "name": name, "text": text, "meta": {"id":tid, "kind":"type","name":name}}

def generation_doc(g):
    gid=g["id"]; name=g["name"]; region=(g.get("main_region") or {}).get("name")
    text=f"{name} — main region: {region}"
    return {"id": f"generation:{gid}", "name": name, "text": text, "meta":{"id":gid,"kind":"generation","region":region}}

def region_doc(r):
    rid=r["id"]; name=r["name"]
    locs = [l["name"] for l in r.get("locations", [])[:50]]
    text=f"Region {name} — notable locations: {', '.join(locs)}"
    return {"id": f"region:{rid}", "name": name, "text": text, "meta":{"id":rid,"kind":"region"}}


def _choose_key_moves(poke_json, types):
    stabs = {t["type"]["name"] for t in poke_json["types"]}
    # filter for "level-up moves"
    moves = []
    for m in poke_json["moves"]:
        md = [d for d in m["version_group_details"] if d["move_learn_method"]["name"] == "level-up"]
        if not md: continue
        moves.append(m["move"]["name"])

    # rank by: STAB > known/iconic names > power (requires per-move fetch)
    ranked = []
    for mv in moves[:40]:  # cap network calls
        try:
            mj = _get_or_none(f"{BASE}/move/{mv}")
            power = mj.get("power") or 0
            mtype = mj["type"]["name"]
            stab = 1 if mtype in stabs else 0
            iconic = 1 if mv in ICONIC_MOVES else 0
            ranked.append((stab, iconic, power, mj["name"]))
        except Exception:
            continue
    ranked.sort(reverse=True)
    out = [mv for _,_,_,mv in ranked[:3]]
    # Title-case nicely
    return [s.replace("-", " ").title() for s in out] or ["Tackle"]

def fetch_pokemon_features(name: str):
    slug = name.strip().lower()
    # --- Use the universal resolver ---
    s, p, species_name, default_form = resolve_species_and_default_form_sync(slug)

    # Types come from the (default) Pokémon form
    types = [t["type"]["name"].replace("-", " ").title()
             for t in sorted(p.get("types", []), key=lambda x: x.get("slot", 99))]

    # Height/weight: PokéAPI uses decimeters/hectograms
    height_m = _meters(p.get("height"))   # decimeters -> meters
    weight_kg = _kg(p.get("weight"))      # hectograms -> kilograms
    size = _size_class(height_m)

    # Region from species.generation (map via your dict)
    gen_name = (s.get("generation") or {}).get("name")
    region = GEN_TO_REGION.get(gen_name, "Unknown")

    # Shape/color/genus/flavor from SPECIES
    shape = (s.get("shape") or {}).get("name")
    color = (s.get("color") or {}).get("name")

    genus = next((g["genus"] for g in s.get("genera", [])
                  if g.get("language", {}).get("name") == "en"), "") or ""
    # Your NLTK narrowing (keep as-is if you want the noun head)
    if genus:
        isNoun = lambda pos: pos[:2] == 'NN'
        tokenized = nltk.word_tokenize(genus.replace("Pokémon", "").strip())
        nouns = [word for (word, pos) in nltk.pos_tag(tokenized) if isNoun(pos)]
        genus = nouns[0].lower() if nouns else genus.replace("Pokémon", "").strip().lower()
    else:
        genus = ""

    flavor_en = " ".join(
        ft.get("flavor_text","").replace("\n", " ").replace("\f", " ")
        for ft in s.get("flavor_text_entries", [])
        if (ft.get("language") or {}).get("name") == "en"
    )

    # Abilities (prefer non-hidden, then by slot)
    abilities_sorted = sorted(p.get("abilities", []),
                              key=lambda a: (a.get("is_hidden", False), a.get("slot", 999)))
    abilities = [a["ability"]["name"].replace("-", " ").title() for a in abilities_sorted]

    key_moves = _choose_key_moves(p, types)  # your existing heuristic

    return {
        "name": species_name.replace("-", " ").title(),   # canonical species display name
        "region": region,
        "types": types,                # e.g., ["Water","Psychic"]
        "size": size,                  # small / medium / large
        "height_m": height_m,
        "weight_kg": weight_kg,
        "shape": shape,                # 'quadruped', 'fish', 'upright', ...
        "color": color,                # 'pink', 'blue', ...
        "genus": genus,                # e.g., 'Seed' (noun head if NLTK enabled)
        "flavor_text": flavor_en,      # concatenated English flavor text entries
        "is_legendary": s.get("is_legendary", False) or s.get("is_mythical", False),
        "is_mythical": s.get("is_mythical", False),
        "key_moves": key_moves,        # 2–3 iconic/typed moves
        "ability": abilities[0] if abilities else None,
        "sources": {
            "pokemon": f"{BASE}/pokemon/{default_form}",
            "species": f"{BASE}/pokemon-species/{species_name}"
        }
    }

class PokemonArgs(BaseModel):
    pokemon_name: str

@tool("web_pokemon_features", args_schema=PokemonArgs)
def web_pokemon_features_tool(pokemon_name: str) -> dict:
    '''Collect standard features (name, region, types, size, ...) from an api'''
    return fetch_pokemon_features(pokemon_name)

@lru_cache(maxsize=2048)
def pokemon_lookup(name: str):
    # your tool now accepts {"pokemon_name": name}
    feat = web_pokemon_features_tool.invoke({"pokemon_name": name})
    # normalize into the shape your extractor expects
    return {"meta": {
        "height_m":  feat.get("height_m"),
        "weight_kg": feat.get("weight_kg"),
        "height_dm": feat.get("height_dm"),  # harmless if None
        "weight_hg": feat.get("weight_hg"),
    }}


if BUILD_INDEX:
    # Query the Pokemon API and load the corpus into docs
    # Index species (fast), pokemon (heavy), moves/abilities/types/generations/regions (medium)
    species_idx = list_all("pokemon-species")
    species = pull_many(species_idx, namer=lambda d: f"species_{d['id']}")
    species_map = {s["name"]: {"is_mythical":s["is_mythical"],"is_legendary":s["is_legendary"],"generation":(s.get("generation") or {}).get("name")} for s in species}

    moves = pull_many(list_all("move")[:400], namer=lambda d: f"move_{d['id']}")           # cap first to keep it snappy
    abilities = pull_many(list_all("ability")[:300], namer=lambda d: f"ability_{d['id']}")
    types = pull_many(list_all("type"), namer=lambda d: f"type_{d['id']}")
    gens  = pull_many(list_all("generation"), namer=lambda d: f"generation_{d['id']}")
    regions = pull_many(list_all("region"), namer=lambda d: f"region_{d['id']}")

    docs = []
    docs += [species_doc(s, pokemon_lookup=pokemon_lookup) for s in species]
    docs += [move_doc(m) for m in moves]
    docs += [ability_doc(a) for a in abilities]
    docs += [type_doc(t) for t in types]
    docs += [generation_doc(g) for g in gens]
    docs += [region_doc(r) for r in regions]

    # (Optional later) Pokémon proper (heavy): uncomment when ready
    # pokemon = pull_many(list_all("pokemon")[:300], namer=lambda d: f"pokemon_{d['id']}")
    # docs += [pokemon_doc(p, species_map) for p in pokemon]
!

### Load documents into a FAISS Vector Store to enable Similarity Search

In [5]:
# Index and embed the data and set up the FAISS tool
def embed(texts):
    vecs = []
    for t in texts:
        r = requests.post(OLLAMA_URL, json={"model": EMBED_MODEL, "prompt": t}, timeout=60)
        r.raise_for_status()
        v = r.json()["embedding"]
        vecs.append(np.array(v, dtype=np.float32))
    return np.vstack(vecs)

def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True) + 1e-9
    return v / n

def build_faiss(docs, path=INDEX_PATH):
    ''' Sets up the FAISS (Facebook AI Similarity Search) tool to perform similarity search and vector clustering'''
    texts = [d["text"] for d in docs]
    X = normalize(embed(texts))
    index = faiss.IndexFlatIP(X.shape[1])
    index.add(X)
    faiss.write_index(index, f"{path}.index")
    with open(f"{path}.meta.json","w") as fp:
        json.dump(docs, fp)
    return path

def load_faiss(path=INDEX_PATH):
    index = faiss.read_index(f"{path}.index")
    meta  = json.load(open(f"{path}.meta.json"))
    return index, meta

if BUILD_INDEX:
    # Perform the ETL -- do the FAISS indexing and metadata generation
    path = build_faiss(docs)
    index, meta = load_faiss(path)

### Establish the data retrieval and generation capability

In [6]:
# Define the PokeRAG data class for querying
class PokeRAG:
    def __init__(self, index_path: str = INDEX_PATH):
        idx_file = f"{index_path}.index"
        meta_file = f"{index_path}.meta.json"
        if not (os.path.exists(idx_file) and os.path.exists(meta_file)):
            raise FileNotFoundError(
                f"Missing index/meta. Expected {idx_file} and {meta_file}. "
                "Build them with your ETL/index step first."
            )
        self.index = faiss.read_index(idx_file)
        with open(meta_file, "r") as fp:
            self.meta = json.load(fp)

    def _filter(self, d: Dict[str, Any], *, 
                kind: Optional[str], mythical_only: bool,
                types: Optional[List[str]], region: Optional[str],
                generation: Optional[str], shape: Optional[str]) -> bool:
        m = d.get("meta", {})
        if kind and m.get("kind") != kind:
            return False
        if mythical_only and not m.get("is_mythical"):
            return False
        if types:
            doc_types = {t.lower() for t in (m.get("types") or [])}
            if not set(t.lower() for t in types) <= doc_types:
                return False
        if region and (m.get("region") or m.get("main_region")) != region:
            return False
        if generation and (m.get("generation")) != generation:
            return False
        if shape and ((m.get("shape") or "").lower() != shape.lower()):   
            return False
        return True

    def search(self, query: str, k: int = 8, *, 
               kind: Optional[str] = None, mythical_only: bool = False,
               types: Optional[List[str]] = None, region: Optional[str] = None,
               generation: Optional[str] = None, shape: Optional[str] = None) -> List[Dict[str, Any]]:
        qv = embed([query])
        D, I = self.index.search(qv, max(k * 5, k))  # overfetch then filter
        hits = []
        for i, score in zip(I[0], D[0]):
            if i < 0: 
                continue
            d = self.meta[i]
            if self._filter(d, kind=kind, mythical_only=mythical_only,
                            types=types, region=region, generation=generation, shape=shape):
                out = {
                    "id": d["id"],
                    "name": d.get("name"),
                    "kind": d["meta"].get("kind"),
                    "score": float(score),
                    "snippet": (d.get("text","")[:240] + "…") if len(d.get("text","")) > 240 else d.get("text",""),
                    "meta": d.get("meta", {}),
                }
                hits.append(out)
                if len(hits) >= k:
                    break
        return hits

    # Convenience wrapper returning a dict (nice for tools)
    def search_dict(self, **kwargs) -> Dict[str, Any]:
        return {"results": self.search(**kwargs)}
    
POKE_RAG = PokeRAG(index_path=INDEX_PATH)


In [7]:

def _table_for(hits, kind_lc):
    if (kind_lc or (hits and hits[0].get("kind"))) == "generation":
        cols = ["Generation", "Region"]
        rows = []
        for h in hits:
            gid = h.get("meta",{}).get("id")
            reg = _cap(h.get("meta",{}).get("region"))
            rows.append([f"Generation {_roman(gid)}", reg])
    else:
        cols = ["Name", "Kind"]
        rows = [[_cap(h.get("name")), h.get("kind")] for h in hits]
    # markdown + very simple HTML (fallback-safe)
    md = "| " + " | ".join(cols) + " |\n|"+ "|".join(["---"]*len(cols)) + "|\n" + \
         "\n".join("| " + " | ".join(map(str,row)) + " |" for row in rows)
    html = "<table>" + \
           "<thead><tr>" + "".join(f"<th>{c}</th>" for c in cols) + "</tr></thead>" + \
           "<tbody>" + "".join("<tr>" + "".join(f"<td>{c}</td>" for c in row) + "</tr>" for row in rows) + \
           "</tbody></table>"
    return cols, rows, md, html

class PokeRAGArgs(BaseModel):
    # query / filters
    query: str = Field("", description="Natural-language question or keywords.")
    k: int = Field(8, ge=1, le=9999)
    kind: Optional[str] = Field(None, description="species|pokemon|move|ability|type|generation|region")
    mythical_only: bool = False
    types: Optional[List[str]] = None
    region: Optional[str] = None
    generation: Optional[str] = None
    shape: Optional[str] = Field(None, description="Filter species by PokéAPI shape slug, e.g., 'quadruped', 'fish'.")
    require_all_types: bool = Field(False, description="If true, doc must include ALL listed types (AND).")
    # formatting
    view: Literal["raw","names","pairs","text","table","dataframe","html"] = "raw"
    distinct: bool = Field(True, description="Deduplicate by name in formatted views.")
    # retrieval mode
    mode: Literal["search","list"] = Field("search", description="list = deterministic scan by kind; search = vector search")

@tool("poke_rag_search", args_schema=PokeRAGArgs)
def poke_rag_search(
    query: str = "",
    k: int = 8,
    kind: Optional[str] = None,
    legendary_only: bool = False,
    mythical_only: bool = False,
    types: Optional[List[str]] = None,
    region: Optional[str] = None,
    generation: Optional[str] = None,
    shape: Optional[str] = None,
    require_all_types: bool = False,
    view: str = "raw",
    distinct: bool = True,
    mode: str = "search",
) -> Dict[str, Any]:
    """Search local Poké RAG (FAISS + metadata). Use mode='list' for complete enumerations (e.g., generations)."""

    # ---- shared filter ----
    want_types = {t.lower() for t in (types or [])}
    reg_lc = (region or "").lower()
    gen_lc = (generation or "").lower()
    kind_lc = (kind or "").lower()

    def _passes(d: Dict[str, Any]) -> bool:
        m = d.get("meta", {})
        if kind and (m.get("kind","").lower() != kind_lc): return False
        if legendary_only and not m.get("is_legendary"): return False
        if mythical_only and not m.get("is_mythical"): return False
        if types:
            doc_types = {t.lower() for t in (m.get("types") or [])}
            if require_all_types:
                if not (want_types <= doc_types): return False  # AND
            else:
                if not (want_types & doc_types): return False    # OR
        if region and ((m.get("region") or m.get("main_region") or "").lower() != reg_lc): return False
        if generation and ((m.get("generation") or "").lower() != gen_lc): return False
        if shape and ((m.get("shape") or "").lower() != shape.lower()): return False 
        return True

    # ---- retrieval ----
    if mode == "list" and kind:
        # Deterministic scan of all docs of this kind (no vector search).
        items = [d for d in POKE_RAG.meta if (d.get("meta",{}).get("kind","").lower() == kind_lc)]
        items = [d for d in items if _passes(d)]
        # sort: by numeric meta.id if present, else by name
        def sort_key(d):
            mid = d.get("meta",{}).get("id")
            nm  = d.get("name") or ""
            return (0, mid) if isinstance(mid, int) else (1, nm)
        items.sort(key=sort_key)
        hits = [{
            "id": d["id"],
            "name": d.get("name"),
            "kind": d.get("meta",{}).get("kind"),
            "score": 1.0,
            "snippet": (d.get("text","")[:240] + "…") if len(d.get("text",""))>240 else d.get("text",""),
            "meta": d.get("meta",{}),
        } for d in items[:k]]
    else:
        # Vector search first, then filter; top up with list-mode if kind is set and results are sparse.
        hits = POKE_RAG.search(
            query=query or kind or "pokemon",
            k=max(k*5, k),  # overfetch for filtering
            kind=None, mythical_only=False, types=None, region=None, generation=None,
            shape=shape
        )
        hits = [h for h in hits if _passes(h)]
        # Top-up to k with deterministic list of same kind (if provided)
        if kind and len(hits) < k:
            seen = {(h.get("name") or "").lower() for h in hits}
            pool = [d for d in POKE_RAG.meta
                    if (d.get("meta",{}).get("kind","").lower() == kind_lc)
                    and _passes(d)
                    and ((d.get("name") or "").lower() not in seen)]
            def sort_key(d):
                mid = d.get("meta",{}).get("id")
                nm  = d.get("name") or ""
                return (0, mid) if isinstance(mid, int) else (1, nm)
            pool.sort(key=sort_key)
            for d in pool[:(k - len(hits))]:
                hits.append({
                    "id": d["id"], "name": d.get("name"),
                    "kind": d.get("meta",{}).get("kind"),
                    "score": 0.0,
                    "snippet": (d.get("text","")[:240] + "…") if len(d.get("text",""))>240 else d.get("text",""),
                    "meta": d.get("meta",{}),
                })
        # trim to k
        hits = hits[:k]

    # ---- formatting ----
    if not hits or view == "raw":
        return {"results": hits}

    # stable sort again for deterministic formatting
    def sort_key_fmt(h):
        mid = h.get("meta",{}).get("id")
        nm  = h.get("name") or ""
        return (0, mid) if isinstance(mid, int) else (1, nm)
    hits.sort(key=sort_key_fmt)

    # dedupe by name if requested
    if distinct:
        seen=set(); tmp=[]
        for h in hits:
            nm = (h.get("name") or "").lower()
            if nm in seen: continue
            seen.add(nm); tmp.append(h)
        hits = tmp

    if view == "names":
        items = []
        for h in hits:
            if (kind_lc or h.get("kind")) == "generation":
                gid = h.get("meta",{}).get("id")
                items.append(f"Generation {_roman(gid) if isinstance(gid,int) else _cap(h.get('name'))}")
            else:
                items.append(_cap(h.get("name")))
        return {"items": items}

    if view == "pairs":
        rows = []
        for h in hits:
            knd = (kind_lc or h.get("kind"))
            if knd == "generation":
                gid = h.get("meta",{}).get("id")
                reg = _cap(h.get("meta",{}).get("region"))
                rows.append({"generation": f"Generation {_roman(gid) if isinstance(gid,int) else _cap(h.get('name'))}",
                             "region": reg})
            elif knd in {"species","pokemon"}:
                rows.append({"name": _cap(h.get("name")),
                             "types": [_cap(t) for t in (h.get("meta",{}).get("types") or [])],
                             "shape": _cap(h.get("meta",{}).get("shape") or ""), })
            else:
                rows.append({"name": _cap(h.get("name")), "kind": knd})
        return {"rows": rows}

    if view == "text":
        lines=[]
        for h in hits:
            knd = (kind_lc or h.get("kind"))
            if knd == "generation":
                gid = h.get("meta",{}).get("id")
                reg = _cap(h.get("meta",{}).get("region"))
                lines.append(f"- Generation {_roman(gid)} — {reg}")
            elif knd in {"species","pokemon"}:
                types_list = [_cap(t) for t in (h.get("meta",{}).get("types") or [])]
                lines.append(f"- {_cap(h.get('name'))}" + (f" — {', '.join(types_list)}" if types_list else ""))
            else:
                lines.append(f"- {_cap(h.get('name'))}")
        return {"text": "\n".join(lines)}

    if view in ("table","dataframe","html"):
        cols, rows, md, html = _table_for(hits, kind_lc)
        out = {"columns": cols, "rows": rows}
        if view == "table":
            out["markdown"] = md
        if view == "html":
            out["html"] = html
        if view == "dataframe":
            out["table"] = out  # explicit table payload for UIs
        return out

    # fallback
    return {"results": hits}

def _find_entity_raw(name: str, kinds=("species","pokemon")) -> dict | None:
    nm = _norm(name); best = None
    for knd in kinds:
        raw = poke_rag_search.invoke({"query": name, "kind": knd, "mode": "search", "view": "raw", "k": 6})
        pool = raw.get("results", []) or []
        exact = [d for d in pool if _norm(d.get("name")) == nm]
        if exact: return exact[0]
        best = best or (pool[0] if pool else None)
    if best: return best
    for knd in kinds:
        lst = poke_rag_search.invoke({"kind": knd, "mode": "list", "view": "raw", "k": 5000})
        for d in lst.get("results", []) or []:
            if _norm(d.get("name")) == nm:
                return d
    return None

def _find_default_pokemon_for(name: str) -> dict | None:
    nm = _norm(name)

    # Try search first (k a bit higher)
    raw = poke_rag_search.invoke({"query": name, "kind": "pokemon", "mode": "search", "view": "raw", "k": 25}) or {}
    pool = raw.get("results", []) or []

    # Prefer exact-name or meta.is_default
    exact = [d for d in pool if _norm(d.get("name")) == nm]
    if exact:
        # among exacts, pick default if flagged
        ex_default = [d for d in exact if (d.get("meta", {}) or {}).get("is_default")]
        return (ex_default[0] if ex_default else exact[0])

    # Fall back to list scan if search was sparse
    lst = poke_rag_search.invoke({"kind": "pokemon", "mode": "list", "view": "raw", "k": 5000}) or {}
    cands = [d for d in (lst.get("results", []) or []) if _norm(d.get("name")) == nm]
    if cands:
        c_default = [d for d in cands if (d.get("meta", {}) or {}).get("is_default")]
        return (c_default[0] if c_default else cands[0])

    # As a last resort, take first from search pool
    return pool[0] if pool else None




In [8]:

Operation = Literal["COUNT","LIST","ATTRIBUTE","ENUM_VALUES","QA"]
Kind      = Literal["species","pokemon","move","ability","generation","region","type"]
Attr      = Literal["types","shape","region","generation","height_m","weight_kg",
                    "power","accuracy","pp","damage_class","effect_text",
                    "is_mythical","is_legendary"]
FieldName = Literal["name","types","shape","region","generation"]

class Filters(BaseModel):
    types: Optional[List[str]] = None
    require_all_types: bool = True
    shape: Optional[str] = None          # e.g. "quadruped"
    region: Optional[str] = None         # e.g. "johto"
    generation: Optional[str] = None     # e.g. "generation-ii"
    mythical_only: bool = False
    legendary_only: bool = False

class QueryPlan(BaseModel):
    op: Operation
    kind: Optional[Kind] = None          # default "species" for most list/enum
    entity: Optional[str] = None         # for ATTRIBUTE (e.g., "Slowpoke")
    attribute: Optional[Attr] = None     # for ATTRIBUTE (e.g., "shape")
    fields: Optional[List[FieldName]] = None  # for LIST tables (columns to return)
    filters: Filters = Field(default_factory=Filters)
    k: int = 500

def _enum_values(field: str) -> dict:
    raw = poke_rag_search.invoke({"kind":"species","mode":"list","view":"raw","k":5000})
    vals = []
    for d in raw.get("results", []) or []:
        m = d.get("meta", {}) or {}
        v = m.get(field)
        if not v and field == "region":
            v = GEN_TO_REGION.get((m.get("generation") or "").lower())
        if isinstance(v, list):
            vals.extend(str(x).lower() for x in v if x)
        elif isinstance(v, str):
            vals.append(v.lower())
    uniq = sorted(set(vals))
    pretty = [u.replace("-", " ").title() for u in uniq]
    label = field if field.endswith("s") else f"{field}s"
    md = "| Value |\n|---|\n" + "\n".join(f"| {v} |" for v in pretty)
    return {
        "answer": f"{len(uniq)} {label}: " + ", ".join(pretty) + ".",
        "table_markdown": md,
        "evidence": {"values": uniq}
    }

### Build the JSON query planner to turn a user query into an LLM input.

In [9]:


PLAN_PROMPT = f"""You are a strict JSON query planner for a Pokémon QA system.
Map the question to a QueryPlan. Output ONLY JSON. No explanations.

Rules:
- Use op ENUM_VALUES when the user asks for “possible values” of a field (e.g., shapes).
  Set kind="species" and fields=["shape"] and ignore k.
- Use op COUNT when the user asks “how many …”.
- Use op LIST for “what/which/list/show … Pokémon …” with filters (types/region/shape/legendary/mythical).
  Default kind="species". Use filters.require_all_types=true for multi-type filters.
- Use op ATTRIBUTE for “what is the X of Y?” (e.g., Slowpoke shape, Flamethrower power).
- Otherwise use op QA.
- “how many <field plural>” → COUNT with attribute set to that field (e.g., shapes/types/regions/generations).
- For LIST of Pokémon with filters (types/region/shape/legendary/mythical), set kind="species".


Allowed shapes: {', '.join(list(SHAPES))}.
Allowed regions: {', '.join(list(REGION_TO_GEN.keys()))}.
Types: {', '.join(list(TYPE_SET))}.

Question: {{question}}
"""


def _clean(s: str) -> str:
    # normalize curly quotes, em-dashes, etc.
    s = unicodedata.normalize("NFKC", s)
    return s


In [10]:
def _find_move_doc_by_name(name: str) -> dict | None:
    """Exact-name first from the full local catalog; fallback to vector search."""
    nm = _norm(name)
    # list-mode: scan all moves deterministically
    lst = poke_rag_search.invoke({"kind": "move", "mode": "list", "view": "raw", "k": 5000})
    for d in lst.get("results", []):
        if _norm(d.get("name")) == nm:
            return d
    # fallback: vector
    raw = poke_rag_search.invoke({"query": name, "kind": "move", "mode": "search", "view": "raw", "k": 6})
    return (raw.get("results") or [None])[0]

def _read_move_effect(doc: dict) -> str | None:
    if not doc: return None
    m = (doc.get("meta") or {})
    eff = m.get("effect_text")
    if eff: return eff.strip()
    blob = " ".join([doc.get("snippet") or "", doc.get("text") or ""]).strip()
    if not blob: return None
    m1 = re.search(r"Effect:\s*(.+?)(?:\s*(?:Power|Accuracy|PP|Type|Class)\b|$)", blob, re.I)
    if m1: return m1.group(1).strip()
    m2 = re.search(r"(Has a .*? chance .*?\.|Burns the target\.|Deals .*? damage\.)", blob, re.I)
    return m2.group(1).strip() if m2 else None

def _read_move_stat(doc: dict, field: str):
    """
    field in {'power','accuracy','pp','damage_class'}
    Returns a normalized Python value or None if unknown.
    """
    if not doc: return None
    m = (doc.get("meta") or {})
    # 1) Prefer structured meta if present
    if field in {"power","accuracy","pp"} and m.get(field) is not None:
        return m[field]
    if field == "damage_class" and m.get(field):
        return str(m[field]).title()

    # 2) Parse from text/snippet if meta missing
    blob = " ".join([doc.get("snippet") or "", doc.get("text") or ""]).strip()
    if not blob: return None

    if field == "power":
        mm = re.search(r"\bPower:\s*(\d+)\b", blob, re.I)
        return int(mm.group(1)) if mm else None

    if field == "accuracy":
        mm = re.search(r"\bAccuracy:\s*(\d+)\b", blob, re.I)
        # PokéAPI sometimes uses null for “never misses”
        return int(mm.group(1)) if mm else "Never misses" if re.search(r"never misses", blob, re.I) else None

    if field == "pp":
        mm = re.search(r"\bPP:\s*(\d+)\b", blob, re.I)
        return int(mm.group(1)) if mm else None

    if field == "damage_class":
        mm = re.search(r"\b(Damage\s*Class|Class):\s*(Physical|Special|Status)\b", blob, re.I)
        return mm.group(2).title() if mm else None

    return None


In [11]:
# Takes in a QueryPlan and question.
def execute(plan: QueryPlan, question: str | None = None) -> dict:
    # ENUM_VALUES (e.g., shapes)
    if plan.op == "ENUM_VALUES":
        field = normalize_attr((plan.fields or ["shape"])[0])
        return _enum_values(field)
    # COUNT
    if plan.op == "COUNT":
        attr = normalize_attr(plan.attribute)
        if attr in {"shape","types","region","generation"}:
            sub = _enum_values(attr)
            n = len(sub["evidence"]["values"])
            label = attr if attr.endswith("s") else f"{attr}s"
            return {"answer": f"{n} {label}."}
        list_plan = plan.model_copy(); list_plan.op = "LIST"
        rows = execute(list_plan, question).get("rows", [])
        return {"answer": f"{len(rows)}."}

    # LIST (facet search over species)
    if plan.op == "LIST":
        f = plan.filters
        region = (f.region or "").lower()
        generation = f.generation or (REGION_TO_GEN.get(region) if region else None)

        raw = poke_rag_search.invoke({"kind":"species","mode":"list","view":"raw","k":5000})
        rows = []
        for d in raw.get("results", []) or []:
            m = d.get("meta",{}) or {}

            if f.legendary_only and not m.get("is_legendary"): continue
            if f.mythical_only  and not m.get("is_mythical"):  continue
            if f.shape and (m.get("shape") or "").lower() != f.shape.lower(): continue

            if f.types:
                have = {t.lower() for t in (m.get("types") or [])}
                need = {t.lower() for t in f.types}
                if f.require_all_types:
                    if not (need <= have): continue
                else:
                    if not (need & have): continue

            if generation:
                reg = (m.get("region") or m.get("main_region") or "").lower()
                gen = (m.get("generation") or "").lower()
                if reg:
                    if reg != region: continue
                else:
                    if gen != generation: continue

            rows.append({
                "name": _cap(d.get("name")),
                "types": [_cap(t) for t in (m.get("types") or [])],
                "region": _cap(reg or GEN_TO_REGION.get(gen, "")) if (reg:= (m.get("region") or m.get("main_region") or "").lower()) or (gen:= (m.get("generation") or "").lower()) else "—",
                "shape": _cap(m.get("shape") or ""),
            })

        answer = ("None found." if not rows
                  else (", ".join([r["name"] for r in rows[:10]]) + (f", and {len(rows)-10} more." if len(rows)>10 else "")))
        return {"answer": answer, "rows": rows}
    # ATTRIBUTE (structured lookups; you already have helpers for moves/effects/stats)
    if plan.op == "ATTRIBUTE":
        attr = normalize_attr(plan.attribute)
        ent  = (plan.entity or "").strip()
        if not ent: return {"answer":"No entity specified."}

        # Moves (effect/power/accuracy/pp/damage_class)
        if attr in {"effect_text","power","accuracy","pp","damage_class"}:
            doc = _find_move_doc_by_name(ent)
            if not doc: return {"answer":"Not found."}
            if attr == "effect_text":
                eff = _read_move_effect(doc)
                return {"answer": eff or "Effect not specified in the context.", "evidence": doc}
            val = _read_move_stat(doc, attr)
            if val is not None:
                name = (doc.get('name') or ent).replace("-"," ").title()
                label = "damage class" if attr=="damage_class" else attr.replace("_"," ")
                suffix = "%" if (attr=="accuracy" and isinstance(val,int)) else ""
                return {"answer": f"{name} {label}: {val}{suffix}.", "evidence": doc}
            return {"answer": "Not specified in the context.", "evidence": doc}

        # Species/pokemon (shape/types/region/generation/height/weight)
        doc = _find_entity_raw(ent, kinds=("species","pokemon"))
        if not doc: return {"answer":"Not found."}
        m = (doc.get("meta") or {})
        name = (doc.get("name") or ent).replace("-"," ").title()
        if attr == "shape":
            shp = (m.get("shape") or "")
            return {"answer": f"{name} is {_cap(shp)}." if shp else "Shape not available.", "evidence": doc}
        if attr == "types":
            tt = [_cap(t) for t in (m.get("types") or [])]
            return {"answer": f"{name} {'is' if len(tt)==1 else 'are'} " + " / ".join(tt) + "." if tt else f"Types not available.", "evidence": doc}
        if attr == "region":
            reg = (m.get("region") or m.get("main_region") or GEN_TO_REGION.get((m.get("generation") or "").lower(),"") or "")
            return {"answer": f"{name}: Region {_cap(reg)}." if reg else "Region not available.", "evidence": doc}
        if attr == "generation":
            gen = (m.get("generation") or "")
            return {"answer": f"{name} appears in Generation {gen.split('-',1)[1].upper()}." if gen else "Generation not available.", "evidence": doc}
        def _pick_metric(doc, attr):
            """Read height/weight regardless of how ETL named/stored it; do unit fixes."""
            if not doc:
                return None
            m = (doc.get("meta") or {}) or {}

            # direct hit
            v = m.get(attr)
            if v is not None:
                return float(v)

            # common alternates + conversions
            if attr == "height_m":
                # PokeAPI: height in decimeters
                if m.get("height_dm") is not None:
                    return float(m["height_dm"]) / 10.0
                if m.get("height") is not None:  # already meters in some ETLs
                    return float(m["height"])
            if attr == "weight_kg":
                # PokeAPI: weight in hectograms
                if m.get("weight_hg") is not None:
                    return float(m["weight_hg"]) / 10.0
                if m.get("weight") is not None:  # already kg in some ETLs
                    return float(m["weight"])

            return None
        if attr in {"height_m","weight_kg"}:
            # 0) species meta (usually None)
            val = _pick_metric(doc, attr)
            poke_doc = None

            # 1) local pokemon doc (you don’t have these—will be None/False per your debug)
            if val is None:
                poke_doc = _find_default_pokemon_for(plan.entity or "")
                val = _pick_metric(poke_doc, attr)

            # 2) FINAL: web fallback (this is the missing piece)
            if val is None:
                ent_clean = re.sub(r"[^a-z0-9\- ]+", "", (plan.entity or "").lower()).strip()
                try:
                    feat = web_pokemon_features_tool.invoke({"pokemon_name": ent_clean})
                    val = (float(feat.get("height_m")) if attr == "height_m"
                        else float(feat.get("weight_kg")))
                except Exception:
                    val = None

            if val is None:
                return {"answer": "Not available.", "evidence": {"species": doc, "pokemon": poke_doc}}

            pretty = (doc.get("name") or plan.entity or "").replace("-", " ").title()
            if attr == "height_m":
                return {"answer": f"{pretty} is {val:.2f} m tall.", "evidence": {"species": doc, "pokemon": poke_doc}}
            else:
                return {"answer": f"{pretty} weighs {val:.1f} kg.", "evidence": {"species": doc, "pokemon": poke_doc}}
    # -------------------------------------------------------------------
    # ------------------------- QA FALLBACK -----------------------------
    # ----------- Only for questions not already handled ----------------
    # -------------------------------------------------------------------
    def _nl_from_plan(p: QueryPlan) -> str:
        if p.op == "ATTRIBUTE" and p.attribute and p.entity:
            return f"what is the {p.attribute.replace('_',' ')} of {p.entity}?"
        if p.op == "LIST":
            parts = []
            f = p.filters
            if f.types: parts.append("types=" + "/".join(f.types))
            if f.shape: parts.append(f"shape={f.shape}")
            if f.region: parts.append(f"region={f.region}")
            if f.generation: parts.append(f"generation={f.generation}")
            if f.legendary_only: parts.append("legendary")
            if f.mythical_only: parts.append("mythical")
            return "list pokemon " + (" ".join(parts) if parts else "")
        if p.op == "ENUM_VALUES" and p.fields:
            return f"list all values of {p.fields[0]}"
        return "answer the question using the context"

    q_text = question or _nl_from_plan(plan)

    res = poke_rag_search.invoke({
        "query": q_text,
        "kind": plan.kind,
        "mode": "search",
        "view": "text",
        "k": plan.k,
    }) or {}

    ctx = res.get("text")
    if not ctx:
        hits = res.get("results") or []
        ctx = "\n".join(
            f"- {h.get('name','')}: {h.get('snippet','')}"
            for h in hits if isinstance(h, dict)
        ) or ""

    if not ctx.strip():
        return {"answer": "I don’t have enough local context to answer that.", "evidence": res}

    ans = PLANNER.invoke(
        f"Answer concisely using ONLY this context:\n\n{ctx[:6000]}\n\nQuestion: {q_text}"
    ).content.strip()

    return {"answer": ans, "evidence": res}

In [12]:

# Does a regex search to determine if the query is a "how tall is" or "what is the effect" type of question.
def preplan(question: str) -> QueryPlan | None:
    qn = _clean(question)
    ql = qn.lower().strip()

    m = re.search(r"\bhow\s+tall\s+is\s+([a-z0-9' -]+)\??", ql)
    if m:
        return QueryPlan(op="ATTRIBUTE", attribute="height_m", entity=m.group(1).strip())

    m = (re.search(r"\bwhat(?:'s| is)\s+the?\s*effect\s+of\s+([a-z0-9' -]+)\??", ql) or
         re.search(r"\beffect\s+of\s+([a-z0-9' -]+)\??", ql))
    if m:
        return QueryPlan(op="ATTRIBUTE", attribute="effect_text", entity=m.group(1).strip())

    return None

# Uses an LLM to determine the type of query
def plan(question: str) -> QueryPlan:
    for _ in range(2):  # try twice in case of formatting hiccups
        out = PLANNER.invoke(PLAN_PROMPT.format(question=question))
        try:
            return QueryPlan.model_validate_json(out.content)
        except Exception:
            continue
    # ultra-safe fallback
    return QueryPlan(op="QA", kind=None, filters=Filters())

# Takes the initial query or existing QueryPlan object and by regex, converts them into a QueryPlan object
def harden_plan(p: QueryPlan, question: str) -> QueryPlan:
    ql = _clean(question).lower().strip()

    # FORCE: “how many …”
    if re.search(r"\bhow\s+many\b", ql):
        if re.search(r"\bshapes?\b", ql):        p.op, p.attribute = "COUNT", "shape"
        elif re.search(r"\btypes?\b", ql):       p.op, p.attribute = "COUNT", "types"
        elif re.search(r"\bregions?\b", ql):     p.op, p.attribute = "COUNT", "region"
        elif re.search(r"\bgenerations?\b", ql): p.op, p.attribute = "generation"

    # FORCE: counts of distinct enums
    if re.search(r"\bhow\s+many\b", ql):
        if re.search(r"\bshapes?\b", ql):        p.op, p.attribute = "COUNT", "shape"
        elif re.search(r"\btypes?\b", ql):       p.op, p.attribute = "COUNT", "types"
        elif re.search(r"\bregions?\b", ql):     p.op, p.attribute = "COUNT", "region"
        elif re.search(r"\bgenerations?\b", ql): p.op, p.attribute = "generation"

    # FORCE: list distinct values (no counting)
    if re.search(r"\b(list|show|what\s+are|give\s+me)\b", ql) and re.search(r"\bshape\s+values?\b", ql):
        p.op, p.kind, p.fields = "ENUM_VALUES", "species", ["shape"]
    if re.search(r"\b(what\s+are|list|show|give\s+me)\b.*\btypes?\b", ql) and "pokemon" in ql:
        p.op, p.kind, p.fields = "ENUM_VALUES", "species", ["types"]

    # ATTRIBUTE fallbacks when planner misses entity
    m = (re.search(r"\bwhat\s+(?:is\s+)?(?:the\s+)?shape\s+(?:of\s+|is\s+)([a-z0-9' -]+)\??", ql) or
         re.search(r"\bwhat\s+(?:is\s+)?(?:the\s+)?region\s+(?:of\s+|is\s+)([a-z0-9' -]+)\??", ql) or
         re.search(r"\bwhat\s+types?\s+(?:does\s+|does\s+a\s+|is\s+|are\s+)([a-z0-9' -]+)\??", ql) or
         re.search(r"\bhow\s+tall\s+is\s+([a-z0-9' -]+)\??", ql) or
         re.search(r"\bhow\s+much\s+does\s+([a-z0-9' -]+)\s+weigh\??", ql))
    if m:
        ent = re.sub(r"^(?:a|an|the)\s+", "", m.group(1).strip(), flags=re.I)
        p.op = "ATTRIBUTE"
        if   "shape"  in ql: p.attribute = "shape"
        elif "region" in ql: p.attribute = "region"
        elif "types"  in ql or re.search(r"\btype\b", ql): p.attribute = "types"
        elif "tall"   in ql: p.attribute = "height_m"
        elif "weigh"  in ql: p.attribute = "weight_kg"
        p.entity = ent

    # LIST should target species (has types/shape/generation)
    if p.op == "LIST": p.kind = "species"

    # inherit obvious filters
    if p.op in {"LIST","COUNT"}:
        p.filters.types = p.filters.types or detect_types(question)
        if _is_legend(question): p.filters.legendary_only = True
        if _is_mythic(question):  p.filters.mythical_only = True
        rg = detect_region(question)
        if rg and not p.filters.region: p.filters.region = rg

    p.attribute = normalize_attr(p.attribute)
    return p



In [13]:
def ask_poke(q: str):
    # 1) deterministic pre-plan
    p = preplan(q)
    if not p:
        p = plan(q)  # LLM
    # 2) harden
    p = harden_plan(p, q)
    # 3) execute
    return execute(p, q)



### Test RAG system
We test by trying a variety of Pokemon-related questions to evaluate various parts of the system.

In [14]:
from IPython.display import display, Markdown
def check(q):
    try:
        res = ask_poke(q)
        ans = res.get("answer", "")
        ok = bool(ans) and ("None found." not in ans or "how many" in q.lower())
        status = "✅" if ok else "⚠️"
        display(Markdown(f"**{status} Q:** {q}\n\n**A:** {ans}"))
        if "table_markdown" in res: display(Markdown(res["table_markdown"]))
        return ok, res
    except Exception as e:
        display(Markdown(f"**❌ Q:** {q}\n\n**Error:** {e}"))
        return False, {"error": str(e)}

tests = [
    # ENUM_VALUES / COUNT
    "list all shape values",
    "how many pokemon shapes are there?",
    "what are all pokemon types?",
    "how many pokemon regions are there?",
    # LIST (facets)
    "list water pokemon in kanto",
    "list water psychic pokemon",
    "list fire or flying pokemon",
    "list quadruped pokemon from hoenn",
    "list mythical pokemon from sinnoh",
    "what flying legendary pokemon are in the johto region?",
    # ATTRIBUTE
    "what shape is Slowpoke?",
    "what region is Mewtwo from?",
    "what types is Eevee?",
    "how tall is Snorlax?",
    "what is the effect of Flamethrower?",
    "Thunderbolt accuracy?",
    # QA fallback
    "What’s the relationship between Slowpoke and Slowking?"
]
tests = ["what are all pokemon types?"]

results = [check(q)[0] for q in tests]
print(f"Pass: {sum(results)}/{len(tests)}")


**✅ Q:** what are all pokemon types?

**A:** 18 types: Bug, Dark, Dragon, Electric, Fairy, Fighting, Fire, Flying, Ghost, Grass, Ground, Ice, Normal, Poison, Psychic, Rock, Steel, Water.

| Value |
|---|
| Bug |
| Dark |
| Dragon |
| Electric |
| Fairy |
| Fighting |
| Fire |
| Flying |
| Ghost |
| Grass |
| Ground |
| Ice |
| Normal |
| Poison |
| Psychic |
| Rock |
| Steel |
| Water |

Pass: 1/1
